## Theory: Gaussian Elimination

Gaussian Elimination is a direct method used in linear algebra to solve systems of linear equations, find the rank of a matrix, or calculate the inverse of an invertible square matrix. 

A system of $n$ linear equations with $n$ unknown variables can be represented in matrix form as:
$$Ax = b$$

Where:
*   **$A$** is an $n \times n$ coefficient matrix.
*   **$x$** is an $n \times 1$ column vector of unknown variables.
*   **$b$** is an $n \times 1$ column vector of constants.

To solve for $x$, the algorithm applies a sequence of elementary row operations to modify the matrices without changing the solution set. The process is divided into two distinct phases:

### 1. Forward Elimination (Row Echelon Form)
The goal of this phase is to transform the system into an **Augmented Matrix** $M = [A | b]$ and reduce it to an upper triangular matrix (Row Echelon Form). In an upper triangular matrix, all elements below the main diagonal are zero.

This is achieved using elementary row operations:
*   **Swapping rows:** If a diagonal element (the pivot) is zero, we swap that row with a row below it that has a non-zero value in the same column to avoid division by zero.
*   **Row subtraction/addition:** We eliminate the entries below each pivot by subtracting a multiple of the pivot row from the rows below it. The multiplier is defined as $\frac{M_{j,i}}{M_{i,i}}$.

### 2. Back Substitution
Once the matrix is in upper triangular form, the bottom-most equation contains only one unknown variable and can be solved trivially. 

For the $i$-th row, the equation looks like this:
$$M_{i,i}x_i + M_{i,i+1}x_{i+1} + \dots + M_{i,n-1}x_{n-1} = M_{i,n}$$

By substituting the known values of the variables from the bottom up, we can solve for $x_i$:
$$x_i = \frac{M_{i,n} - \sum_{j=i+1}^{n-1} M_{i,j}x_j}{M_{i,i}}$$

## Algorithm: Gaussian Elimination

The following steps outline the logic used in the implementation of the Gaussian Elimination algorithm.

### Phase 1: `row_echlon_form(A, b)`
1. **Validation:** 
   * Check if matrix $A$ is a square matrix (rows = columns).
   * Check if the number of rows in $b$ matches the columns in $A$.
2. **Augmentation:** Create an augmented matrix $M$ by horizontally stacking $A$ and $b$.
3. **Pivoting & Elimination Loop:** For each diagonal element $M[i, i]$ (where $i$ ranges from $0$ to $n-1$):
   * **Zero-Pivot Check:** If the pivot $M[i, i]$ is $0$, search the rows below ($j > i$) for a non-zero entry in the same column. 
     * If found, swap row $i$ with row $j$.
     * If all entries below are $0$, the matrix represents linearly dependent equations (no unique solution). Raise an error.
   * **Row Reduction:** For each row $j$ below the current row $i$:
     * Calculate the scaling factor: $factor = \frac{M[j, i]}{M[i, i]}$.
     * Update the target row by subtracting the scaled pivot row: $Row_j \gets Row_j - (factor \times Row_i)$.
4. Return the upper triangular augmented matrix $M$.

### Phase 2: `back_substituion(Ag_matrix)`
1. **Initialization:** Extract the number of variables $n$ and create a 1D array `solution` of size $n$ initialized with zeros.
2. **Backward Loop:** Iterate backwards from the last row $n-1$ down to $0$:
   * Extract the constant term from the last column of the current row.
   * Calculate the `known_sum` by taking the dot product of the coefficients to the right of the pivot and their correspondingly solved variables in the `solution` array.
   * Solve for the current variable by subtracting the `known_sum` from the constant term, then dividing by the pivot coefficient.
   * Store this value in `solution[i]`.
3. Return the `solution` array containing the values for all unknown variables.


In [25]:
import numpy as np
def row_echlon_form(A,b):
   
    #check is square
    if A.shape[0]!=A.shape[1]:
        raise ValueError ("Expected sqaure matrix")
    if A.shape[1] != b.shape[0]:
        raise ValueError ("Dimension must be same")
    
    #Agmented matrix M
    global M
    M = np.column_stack((A.astype(float),b.astype(float)))
    
    #Number of unknown variable
    n=len(b)
    # Let assum  all coloums are not zeros 
    all_col_zero=False
    
    #Forming row echlon form
    for i in range(n):
        
        if M[i,i]==0:
            #runing for loop to search and swap
            for j in range(i+1,n):
                if M[j,i] != 0:
                    M[[i,j]]=M[[j,i]]
                    break
                else:
                    all_col_zero=True
                    break
            
        if not(all_col_zero):        
           
            pivot=M[i,i]
            #making zero below the pivot
            for j in range(i+1,n):
                #factors of each elements 
                factor = M[j,i]/pivot
                M[j]=M[j]-factor*M[i]     
            
        else:
            raise ValueError ("Colomns are linearly dependent no unique soltuion ")
            break
        
    return M

def back_substituion(Ag_matrix):
    
    num_row,num_col=Ag_matrix.shape
    
    #number of variables to find
    num_var=num_col-1
    
    #defining the 1-D solution matrix with zero entry e.g [x,y,z]
    solution=np.zeros(num_var)
    
    #looping backword to last variable 
    
    for i in range(num_var-1,-1,-1):
        
        #constrant value
        constant=Ag_matrix[i,-1]
        
        # Subtracting known variables already solved in previous iterations
        # matrix[i, i+1:num_vars] fetches the coefficients to the right of the pivot
        # solutions[i+1:num_vars] fetches the answers already calculated
        
        known_sum=np.sum(Ag_matrix[i,i+1:num_var]*solution[i+1:num_var])
        
        #pivat coefficent
        pivat_coeff=Ag_matrix[i,i]
        solution[i]=(constant - known_sum)/pivat_coeff
        
    
    return solution

    
            

A = np.array([[1,1,1],[0,2,5],[2,5,-1]])
b = np.array([6,-4,27])

try: 
    agmented_matrix=row_echlon_form(A,b)
    print("Row echlon form : ","\n",agmented_matrix)
    print("Soltuion Matrix : ","\n",back_substituion(agmented_matrix))
    
except ValueError as e:
    print(f"Error : {e}")

    
    

    
    
    
    

Row echlon form :  
 [[  1.    1.    1.    6. ]
 [  0.    2.    5.   -4. ]
 [  0.    0.  -10.5  21. ]]
Soltuion Matrix :  
 [ 5.  3. -2.]


In [ ]:


solution_numpy = np.linalg.solve(A, b)

print("NumPy solution:", solution_numpy)
print("Our solution   :", back_substituion(agmented_matrix))
print("Both are close ? :", np.allclose(solution_numpy, back_substituion(agmented_matrix)))

NumPy solution: [ 5.  3. -2.]
Our solution   : [ 5.  3. -2.]
Both are close ? : True


### Test-code-block
### try -1

In [ ]:
A=np.array([[0,-2,3],
             [0,-15,0],
             [6,-5,5]   
])

b=np.array([9,-4,17])

M=np.column_stack((A.astype(int),b.astype(int)))

n=len(b)
print(n)
print(A,"\n\n",b,"\n")
print("extract : ",M[[1,2]])


for i in range(n):
    if M[i,i]==0:
        print("zero found :",M)
        for j in range(i+1,n):
            if M[j,i]!=0:
                M[[i,j]]=M[[j,i]]
                print("swaped : ",M)    
                break
    
    pivot=M[i,i]
    print("pivate = ",M[i,i])
    for j in range(i+1,n):
        print("dividing : ",M[j,i])
        factor=M[j,i]/pivot
        print("factor :",factor)
        print("see :",M[j],M[i])
        M[j]=M[j]-factor*M[i]
        print("chnage in M",M)
    
    #print(f"After coulm {i} elimination:\n{M}\n")        
           

### test-code-block 
#### adding conditions

In [ ]:
A=np.array([[0,-2,3],
             [0,-15,0],
             [0,-5,5]   
])

b=np.array([9,-4,17])

M=np.column_stack((A.astype(int),b.astype(int)))
print("A=",A,"\n\n","b = ",b,"\n\n","M=",M)

n=len(b)
all_col_zero=False
for i in range(n):
    #swapping
    if M[i,i]==0:
        
        #runing for loop to search and swap
        for j in range(i+1,n):
            if M[j,i] != 0:
                M[[i,j]]=M[[j,i]]
                break
            else:
                all_col_zero=True
                break
            
    if not(all_col_zero):        
           
        pivot=M[i,i]
        #making zero below the pivot
        for j in range(i+1,n):
            #factors of each elements 
            factor=M[j,i]/pivot
            M[j]=M[j]-factor*M[i]     
            
    else:
        print("Colomns are linearly dependent no unique soltuion ")
        break        
        

      